In [2]:
import numpy as np

For symmetric quantization

1. Scale = max(|xmin|,|xmax|)/127
2. Zp = 0

In [3]:
# calculating symmetric scale and zero point
def symmetric_scale_zero_point(tensor):
    max_abs = np.abs(np.max(tensor))
    min_abs = np.abs(np.min(tensor))
    scale = max(min_abs, max_abs)/127
    zero_point = 0
    return (scale, zero_point)

For asymmetric quantization

1. Scale = (xmax - xmin)/(qmax - qmin)
2. Zero_point = (qmin - (xmin/scale) clip to -128 to 127

In [4]:
# calculating asymmetric scale and zero point
def asymmetric_scale_zero_point(tensor,q_max=127, q_min=-128):
    x_min = np.min(tensor)
    x_max = np.max(tensor)
    x_diff = x_max - x_min
    q_diff = q_max - q_min
    scale = x_diff/q_diff
    zero_point = np.round(q_min - ( x_min / scale)).astype(np.int8)
    zero_point = np.clip(zero_point, q_min, q_max)
    return (scale, zero_point)


In [5]:
# Symmetric Quantization
def symmetric_quantization(tensor,clip=False):
    sym_scale, sym_zero_point = symmetric_scale_zero_point(tensor)
    sym_quant_tensor = np.round((tensor / sym_scale))
    sym_quant_tensor = sym_quant_tensor + sym_zero_point # Here zero point = 0
    sym_quant_clipped = (np.clip(sym_quant_tensor, -127, 127)).astype(np.int8)
    if clip:
        return (sym_quant_clipped, sym_scale, sym_zero_point,sym_quant_tensor)
    else:
        return (sym_quant_clipped, sym_scale, sym_zero_point)

In [6]:
# Asymmetric Quantization
def asymmetric_quantization(tensor,clip=False):
    asym_scale, asym_zero_point = asymmetric_scale_zero_point(tensor)
    asym_quant = np.round((tensor / asym_scale))
    asym_quant = asym_quant + asym_zero_point
    asym_quant_clipped = (np.clip(asym_quant, -128, 127)).astype(np.int8)
    if clip:
        return (asym_quant_clipped, asym_scale, asym_zero_point,asym_quant)
    else:
        return (asym_quant_clipped, asym_scale, asym_zero_point)

In [7]:
# Dequantization
def dequantization_tensor(q_tensor,scale,zero_point):
    dequant = (q_tensor.astype(np.float32) - zero_point)*scale
    return dequant.astype(np.float32)

In [8]:
def quantization_and_performance(tensor,quant_type,t_type,display=True):
    quant_type = quant_type.lower()
    # Symmetric / Asymmetric quantization
    if quant_type == 'symmetric':
        quant_tensor ,scale, zero_point,q_unclip = symmetric_quantization(tensor,clip=True)
    elif quant_type == 'asymmetric':
        quant_tensor ,scale, zero_point,q_unclip = asymmetric_quantization(tensor,clip=True)
    elif quant_type == 'both':
        sym_quant, s_scale, s_zero_point = symmetric_quantization(tensor)
        sym_dequant = dequantization_tensor(sym_quant,s_scale,s_zero_point)
        asym_quant, a_scale, a_zero_point = asymmetric_quantization(tensor)
        asym_dequant = dequantization_tensor(sym_quant,a_scale,a_zero_point)
        display_tensors(tensor,sym_quant,sym_dequant,asym_quant,asym_dequant)
        return
    else:
        print("Unknown quantization type!")
        exit(1)
    #Dequantization
    dequant_tensor= dequantization_tensor(quant_tensor,scale,zero_point)
    #Error and saturation
    abs_error = np.abs(tensor - dequant_tensor)
    max_err = np.max(abs_error)
    mae = np.mean(abs_error)
    mse = np.mean(abs_error**2)
    sat_min = np.sum(q_unclip < -128) # count of values below q_min
    sat_max = np.sum(q_unclip >127) # count of values above q_max
    sat_tot = sat_max + sat_min
    if display:
        display_tensor_and_metrics(t_type,quant_type,tensor,scale,zero_point,quant_tensor,dequant_tensor,mae,mse,max_err, sat_min,sat_max,sat_tot)


In [9]:
def display_tensors(tensor,sym_quant,sym_dequant,asym_quant,asym_dequant):
    print('Original Tensor: \n', tensor)
    print('Symmetric Quant: \n', sym_quant)
    print('Symmetric Dequant: \n', sym_dequant)
    print('Asymmetric Quant: \n', asym_quant)
    print('Asymmetric Dequant: \n', asym_dequant)

In [10]:
def display_tensor_and_metrics(t_type,quant_type,tensor,scale,zero_point,quant_tensor,dequant_tensor,mae,mse,max_err, sat_min,sat_max,sat_tot):
    print(f'Input Type: {t_type}')
    print(f'Quantization Type: {quant_type}')
    print(f'Scale: {scale}')
    print(f'ZeroPoint: {zero_point}')
    print(f'Original Tensor: \n{tensor}')
    print(f'Quantized Tensor: \n{quant_tensor}')
    print(f'Dequantized Tensor: \n{dequant_tensor}')
    print(f'Mean Absolute Error: {mae}')
    print(f'Mean Saturation Error: {mse}')
    print(f'Max Error: {max_err}')
    print(f'Sat Min Error: {sat_min}')
    print(f'Sat Max Error: {sat_max}')
    print(f'Sat Total Error: {sat_tot}')

In [11]:
weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)


In [12]:
#q_type = symmetric / asymmetric
quantization_and_performance(weights, 'symmetric', 'weights')

Input Type: weights
Quantization Type: symmetric
Scale: 0.018897637724876404
ZeroPoint: 0
Original Tensor: 
[[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]
Quantized Tensor: 
[[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]
Dequantized Tensor: 
[[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]
Mean Absolute Error: 0.003700774861499667
Mean Saturation Error: 2.1637999452650547e-05
Max Error: 0.007874011993408203
Sat Min Error: 0
Sat Max Error: 0
Sat Total Error: 0


In [13]:
quantization_and_performance(weights, 'asymmetric', 'weights')

Input Type: weights
Quantization Type: asymmetric
Scale: 0.0172549020498991
ZeroPoint: 11
Original Tensor: 
[[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]
Quantized Tensor: 
[[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]
Dequantized Tensor: 
[[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]
Mean Absolute Error: 0.003803931176662445
Mean Saturation Error: 2.1238029148662463e-05
Max Error: 0.007450997829437256
Sat Min Error: 0
Sat Max Error: 0
Sat Total Error: 0


In [14]:
quantization_and_performance(activations, 'symmetric', 'activations')

Input Type: activations
Quantization Type: symmetric
Scale: 0.025196850299835205
ZeroPoint: 0
Original Tensor: 
[[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]
Quantized Tensor: 
[[  0  12  32  56  83]
 [  4  24  40  71 127]]
Dequantized Tensor: 
[[0.        0.3023622 0.8062992 1.4110236 2.0913386]
 [0.1007874 0.6047244 1.007874  1.7889764 3.2      ]]
Mean Absolute Error: 0.005275561474263668
Mean Saturation Error: 4.482559597818181e-05
Max Error: 0.011023640632629395
Sat Min Error: 0
Sat Max Error: 0
Sat Total Error: 0


In [15]:
quantization_and_performance(activations, 'asymmetric', 'activations')

Input Type: activations
Quantization Type: asymmetric
Scale: 0.012549019418656826
ZeroPoint: -128
Original Tensor: 
[[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]
Quantized Tensor: 
[[-128 -104  -64  -16   39]
 [-120  -80  -48   15  127]]
Dequantized Tensor: 
[[0.         0.30117646 0.80313724 1.4054902  2.0956862 ]
 [0.10039216 0.6023529  1.0039215  1.7945098  3.2       ]]
Mean Absolute Error: 0.0026274309493601322
Mean Saturation Error: 1.1118667316623032e-05
Max Error: 0.0054901838302612305
Sat Min Error: 0
Sat Max Error: 0
Sat Total Error: 0


Since the code calculates the scale and zero point from the same tensor, the tensor's minimum and maximum values are mapped into the valid INT8 range. Therefore, all other values also fall within that range, so no saturation occurs.

In [16]:
quantization_and_performance(activations, 'both', 'activations',display=False)

Original Tensor: 
 [[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]
Symmetric Quant: 
 [[  0  12  32  56  83]
 [  4  24  40  71 127]]
Symmetric Dequant: 
 [[0.        0.3023622 0.8062992 1.4110236 2.0913386]
 [0.1007874 0.6047244 1.007874  1.7889764 3.2      ]]
Asymmetric Quant: 
 [[-128 -104  -64  -16   39]
 [-120  -80  -48   15  127]]
Asymmetric Dequant: 
 [[1.6062745 1.7568628 2.007843  2.3090196 2.6478431]
 [1.6564705 1.9074509 2.1082354 2.4972548 3.2      ]]


In [17]:
quantization_and_performance(weights, 'both', 'weights',display=False)

Original Tensor: 
 [[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]
Symmetric Quant: 
 [[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]
Symmetric Dequant: 
 [[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]
Asymmetric Quant: 
 [[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]
Asymmetric Dequant: 
 [[-1.8290197  -1.0180392  -0.18980393  0.44862744  1.1733333 ]
 [-2.3811765  -0.46588236  0.          0.8109804   1.6392157 ]]


In [18]:
def outliner_symmetric(tensor):
    print("Symmetric Quantization")
    sym_quant, s_scale, s_zero_point = symmetric_quantization(tensor)
    sym_dequant = dequantization_tensor(sym_quant,s_scale,s_zero_point)
    sy_per_ele_err = tensor - sym_dequant
    sy_abs_error = np.abs(sy_per_ele_err)
    sy_mae = np.mean(sy_abs_error)
    sy_mse = np.mean(sy_abs_error**2)
    print('Original Tensor: \n', tensor)
    print(f'Symmetric Quant: \n{sym_quant}')
    print(f'Scale: {s_scale}')
    print(f'ZeroPoint: {s_zero_point}')
    print('Symmetric Dequant: \n', sym_dequant)
    print(f'Mean Absolute Error: {sy_mae}')
    print(f'Mean Saturation Error: {sy_mse}')
    print(f'Per-element Error: \n{sy_per_ele_err}')
    print(f'Max Error: {np.max(sy_abs_error)}')

In [19]:
def outliner_asymmetric(tensor):
    print("Asymmetric Quantization")
    asym_quant, a_scale, a_zero_point = asymmetric_quantization(tensor)
    asym_dequant = dequantization_tensor(asym_quant,a_scale,a_zero_point)
    asy_per_ele_err = tensor - asym_dequant
    asy_abs_error = np.abs(asy_per_ele_err)
    asy_mae = np.mean(asy_abs_error)
    asy_mse = np.mean(asy_abs_error**2)
    print('Asymmetric Quant: \n', asym_quant)
    print(f'Scale: {a_scale}')
    print(f'ZeroPoint: {a_zero_point}')
    print('Asymmetric Dequant: \n', asym_dequant)
    print(f'Mean Absolute Error: {asy_mae}')
    print(f'Mean Saturation Error: {asy_mse}')
    print(f'Per-element Error: \n{asy_per_ele_err}')
    print(f'Max Error: {np.max(asy_abs_error)}')


In [20]:
# outliner
outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0], dtype=np.float32
)

In [21]:
outliner_symmetric(outlier_tensor) # With outliner

Symmetric Quantization
Original Tensor: 
 [-0.5 -0.2  0.   0.3  0.7 12. ]
Symmetric Quant: 
[ -5  -2   0   3   7 127]
Scale: 0.09448818862438202
ZeroPoint: 0
Symmetric Dequant: 
 [-0.47244096 -0.18897638  0.          0.28346455  0.6614173  12.        ]
Mean Absolute Error: 0.01561680156737566
Mean Saturation Error: 0.00044051100849173963
Per-element Error: 
[-0.02755904 -0.01102363  0.          0.01653546  0.03858268  0.        ]
Max Error: 0.038582682609558105


In [22]:
outliner_asymmetric(outlier_tensor)

Asymmetric Quantization
Asymmetric Quant: 
 [-128 -122 -118 -112 -104  127]
Scale: 0.04901960864663124
ZeroPoint: -118
Asymmetric Dequant: 
 [-0.49019608 -0.19607843  0.          0.29411766  0.6862745  12.009804  ]
Mean Absolute Error: 0.007189512252807617
Mean Saturation Error: 7.176663348218426e-05
Per-element Error: 
[-0.00980392 -0.00392157  0.          0.00588235  0.01372546 -0.00980377]
Max Error: 0.013725459575653076


In [23]:
without_outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7], dtype=np.float32
)

In [24]:
outliner_symmetric(without_outlier_tensor)

Symmetric Quantization
Original Tensor: 
 [-0.5 -0.2  0.   0.3  0.7]
Symmetric Quant: 
[-91 -36   0  54 127]
Scale: 0.005511811003088951
ZeroPoint: 0
Symmetric Dequant: 
 [-0.5015748 -0.1984252  0.         0.2976378  0.7      ]
Mean Absolute Error: 0.0011023670667782426
Mean Saturation Error: 2.1080245460325386e-06
Per-element Error: 
[ 0.00157481 -0.0015748   0.          0.00236222  0.        ]
Max Error: 0.0023622214794158936


In [25]:
outliner_asymmetric(without_outlier_tensor)

Asymmetric Quantization
Asymmetric Quant: 
 [-128  -64  -22   42  127]
Scale: 0.004705882631242275
ZeroPoint: -22
Asymmetric Dequant: 
 [-0.49882355 -0.19764706  0.          0.3011765   0.7011765 ]
Mean Absolute Error: 0.0011764795053750277
Mean Saturation Error: 1.9377357602934353e-06
Per-element Error: 
[-0.00117645 -0.00235294  0.         -0.00117648 -0.00117654]
Max Error: 0.0023529380559921265


1. For weights symmetric is preferred as they are equal intervals on both sides
2. Mostly activation uses non-negative values hence by using asymmetric is preferred without wastage of interval
3. The outliner may affect the scale value.
4. By removing outliner the performance is almost similar but the scale differs due to the removal of outline